In [ ]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math
from numba import cuda
import nvtx
import cmasher as cmr
%matplotlib widget

pa.settings.strict_units = False
# A snapshot object
# snap = pa.Snapshot(pa.data_dir, 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/adiabatic-mhd/zoom4_ics_v1/output', 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
# snap = pa.Snapshot('/scratch/lperrone/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
# snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output',247)
snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom8/output', 305, basename='snapshot')

center = snap.Cat.Group['GroupPos'][0]
widths = np.array([5e2, 5e2, 5e2], dtype=float)

# m_filter = 1000*snap.mass
# filter_length = (np.cbrt(3*m_filter/(4*np.pi*snap['0_Density']))).arepo
filter_length = 2.0*snap['0_Diameters']


In [ ]:
# when using the iterative filter we need to make sure that the particles required are
# loaded on the GPU. This can be achieved by increasing the search_radius
# to be larger than the maximum filter_radius of the iterative loop
sf = tc.SmoothingFilter(snap, center, widths, npix=128, orientation=None, 
                        search_radius=10.*filter_length.value)

In [ ]:
sf_spherical = tc.SmoothingFilter(snap, center, widths, npix=128, orientation=None, 
                        search_radius=10.*filter_length.value, tilingType='spherical', rMin=0.0, rMax=4e2)

In [ ]:
filt_density = sf.filter_variable('0_Density',filter_length, 
                                  filter_type="gaussian", iterative=True, optimized=True)

In [ ]:
filt_density_spherical = sf_spherical.filter_variable('0_Density',filter_length,
                                                      filter_type="gaussian", iterative=True)

In [ ]:
# this is to check if the spherical and cartesian tiling and filtering 
# give the same results in the intersecting region

mask_cartesian = np.zeros(snap['0_Density'].shape,dtype='int')
mask_cartesian[sf.index] = sf.isParticleInDomainUnSorted.get()

mask_spherical = np.zeros(snap['0_Density'].shape,dtype='int')
mask_spherical[sf_spherical.index] = sf_spherical.isParticleInDomainUnSorted.get()

snap['0_smooth_var_default'] = snap['0_Density'].copy
snap['0_smooth_var_default'][mask_cartesian>0] = filt_density[sf.isParticleInDomainUnSorted.get()>0]

snap['0_smooth_var_spherical'] = snap['0_Density'].copy
snap['0_smooth_var_spherical'][mask_spherical>0] = filt_density_spherical[sf_spherical.isParticleInDomainUnSorted.get()>0]

mask_tot_inter = mask_cartesian*mask_spherical

np.allclose(snap['0_smooth_var_default'][np.argwhere(mask_tot_inter>0)].value, 
            snap['0_smooth_var_spherical'][np.argwhere(mask_tot_inter>0)].value, 
            rtol=1e-15, atol=1e-8)

In [ ]:
snap['0_DensityTurb'] = snap['0_Density'].copy
snap['0_DensityTurb'][sf.index] = snap['0_Density'][sf.index] - filt_density

snap['0_DensityTurbSpherical'] = snap['0_Density'].copy
snap['0_DensityTurbSpherical'][sf_spherical.index] = snap['0_Density'][sf_spherical.index] - filt_density_spherical

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize

widths_slicer = widths.copy()
widths_slicer[2] = 0.

slicer = pa.Slicer(snap, center, widths_slicer, 'z', npix=512)

dens_tot = slicer.slice_variable('0_Density').to_physical.cgs
dens_turb_cart = slicer.slice_variable('0_DensityTurb').to_physical.cgs
dens_turb_spherical = slicer.slice_variable('0_DensityTurbSpherical').to_physical.cgs

vmin = np.min(dens_tot.value)
vmax = np.max(dens_tot.value)
norm = LogNorm(vmin=vmin,vmax=vmax)
extent = slicer.centered_extent.to('Mpc')

# plt.figure(1)
# plt.clf()

fig, axes = plt.subplots(1,3, figsize=(12,6),sharex=True, sharey=True)

im1 = axes[0].imshow(
    dens_tot.value, origin='lower', norm=norm,extent=extent.value)
im2 = axes[1].imshow(
    dens_turb_cart.value, origin='lower', 
    norm=Normalize(vmin=-0.3*dens_turb_cart.value.max(),vmax=0.3*dens_turb_cart.value.max()),
    cmap="RdBu_r", extent=extent.value)
im3 = axes[2].imshow(
    dens_turb_spherical.value, origin='lower', 
    norm=Normalize(vmin=-0.3*dens_turb_cart.value.max(),vmax=0.3*dens_turb_cart.value.max()),
    cmap="RdBu_r", extent=extent.value)

axes[0].set_title('Total')
axes[1].set_title('Turbulent cartesian')
axes[2].set_title('Turbulent spherical')

for i in range(3):
    axes[i].set_xlabel(extent.label('x'))

axes[0].set_ylabel(extent.label('y'))

# fig.subplots_adjust(bottom=0.2)
# cbar_ax = fig.add_axes([0.1, 0.15, 0.8, 0.05])

cbar = fig.colorbar(im1, orientation='horizontal')
cbar.set_label(dens_tot.label('\\rho'))

cbar = fig.colorbar(im2, orientation='horizontal')
cbar.set_label(dens_tot.label('\\rho'))

cbar = fig.colorbar(im3, orientation='horizontal')
cbar.set_label(dens_tot.label('\\rho'))
# cbar = plt.colorbar()
# fig.tight_layout(pad=1.00, h_pad=None, w_pad=None, rect=None)


fig.subplots_adjust(top=0.9,
bottom=0.1,
left=0.10,
right=0.90,
hspace=0.0,
wspace=0.05)

fig.suptitle('Density field (iterative filter)',y=0.95)


In [ ]:
iteration_converged = -1.0*np.ones(snap['0_Density'].value.shape)
iteration_converged[sf.index] = sf.hasConvergedUnSorted.get()

number_iterations = np.zeros(snap['0_Density'].value.shape)
number_iterations[sf.index] = sf.numIterationsUnSorted.get()

particles_interacted = np.zeros(snap['0_Density'].value.shape)
particles_interacted[sf.index] = sf.hitsNeighboursUnSorted.get()

final_filter_length = 0.0*filter_length.copy
final_filter_length[sf.index] = sf.filter_lengths_out.get()*filter_length.unit

indices_star_forming = np.zeros_like(iteration_converged)
indices_star_forming[snap['0_StarFormationRate'] > 0.] = 1.0

In [ ]:
np.sum(indices_star_forming)

In [ ]:
# slicer = pa.Slicer(snap, center, widths_slicer, 'z', npix=512)

deepproj = pa.Projector(snap, center, np.array([1e3, 1e3, 1e3], dtype=float), 'z', npix=4096)
slicer_iteration_converged = slicer.slice_variable(iteration_converged)
slicer_number_iterations = slicer.slice_variable(number_iterations)
slicer_particles_interacted = slicer.slice_variable(particles_interacted)
slicer_indices_star_forming = deepproj.project_variable(indices_star_forming).value

vmin = 0.0
vmax = 1.0
norm = Normalize(vmin=vmin,vmax=vmax)
extent = slicer.centered_extent.to('Mpc')
cmap=plt.cm.gray
cmap.set_under('yellow')

norm1 = Normalize(vmin=1.0,vmax=np.max(slicer_number_iterations))
cmap1 = plt.cm.inferno
cmap1.set_under('white')

norm2 = LogNorm(vmin=1.0,vmax=np.max(particles_interacted))
cmap2 = plt.cm.inferno
cmap2.set_under('white')

cmap3 = plt.cm.Reds
cmap3.set_under('white',alpha=0.0)

plt.figure(1)
plt.clf()

fig, axes = plt.subplots(1,3, figsize=(12,6),sharex=True, sharey=True)

im1 = axes[0].imshow(
    slicer_iteration_converged, origin='lower', norm=norm,extent=extent.value, cmap=cmap)
im1a = axes[0].imshow(
    slicer_indices_star_forming, origin='lower', norm=norm,extent=extent.value, cmap=cmap3)
im2 = axes[1].imshow(
    slicer_number_iterations, origin='lower', norm=norm1,extent=extent.value, cmap=cmap1)
im3 = axes[2].imshow(
    slicer_particles_interacted, origin='lower', norm=norm2,extent=extent.value, cmap=cmap2)

axes[0].set_title('Has the iteration converged?')
axes[1].set_title('How many iterations it took')
axes[2].set_title('How many particles it interacted')

for i in range(3):
    axes[i].set_xlabel(extent.label('x'))

axes[0].set_ylabel(extent.label('y'))

# fig.subplots_adjust(bottom=0.2)
# cbar_ax = fig.add_axes([0.1, 0.15, 0.8, 0.05])

cbar = fig.colorbar(im1, orientation='horizontal')
cbar.set_label(r'Converged? (1=yes, 0=no)')

cbar = fig.colorbar(im2, orientation='horizontal')
cbar.set_label(r'Iterations')

cbar = fig.colorbar(im3, orientation='horizontal')
cbar.set_label(r'Particles interacted')


fig.subplots_adjust(top=0.9,
bottom=0.1,
left=0.10,
right=0.90,
hspace=0.0,
wspace=0.05)

fig.suptitle('Convergence statistics',y=0.98)


In [ ]:
slicer_indices_star_forming

In [ ]:
np.argwhere(slicer_indices_star_forming>0)

In [ ]:
# slicer = pa.Slicer(snap, center, widths_slicer, 'z', npix=512)

slicer_filter_length = slicer.slice_variable(filter_length)
slicer_final_filter_length = slicer.slice_variable(final_filter_length)

slicer_filter_length = slicer_filter_length.to_physical.astro
slicer_final_filter_length = slicer_final_filter_length.to_physical.astro

relative_delta = (slicer_final_filter_length - slicer_filter_length)/slicer_filter_length

vmin = np.min(slicer_final_filter_length.value)
vmax = np.max(slicer_final_filter_length.value)
norm = LogNorm(vmin=vmin,vmax=vmax)
extent = slicer.centered_extent.to('Mpc')
cmap=plt.cm.inferno
cmap.set_under('white')

vmin = -np.max(np.abs(relative_delta.value))
vmax = -vmin
norm1 = Normalize(vmin=vmin,vmax=vmax)
cmap1 = plt.cm.RdBu_r
cmap1.set_under('white')

plt.figure(1)
plt.clf()

fig, axes = plt.subplots(1,3, figsize=(12,6),sharex=True, sharey=True)

im1 = axes[0].imshow(
   slicer_filter_length.value, origin='lower', norm=norm,extent=extent.value, cmap=cmap)
im2 = axes[1].imshow(
    slicer_final_filter_length.value, origin='lower', norm=norm,extent=extent.value, cmap=cmap)
im3 = axes[2].imshow(
    relative_delta.value, origin='lower', norm=norm1, extent=extent.value, cmap=cmap1)

axes[0].set_title('Pre-determined filter length')
axes[1].set_title('Final filter length')
# axes[2].set_title('How many particles it interacted')

for i in range(3):
    axes[i].set_xlabel(extent.label('x'))

axes[0].set_ylabel(extent.label('y'))

# fig.subplots_adjust(bottom=0.2)
# cbar_ax = fig.add_axes([0.1, 0.15, 0.8, 0.05])

cbar = fig.colorbar(im1, orientation='horizontal')
cbar.set_label(slicer_filter_length.label(r'Filter length'))

cbar = fig.colorbar(im2, orientation='horizontal')
cbar.set_label(slicer_filter_length.label(r'Final filter length'))

cbar = fig.colorbar(im3, orientation='horizontal')
cbar.set_label(relative_delta.label(r'\delta l / l_0'))


fig.subplots_adjust(top=0.9,
bottom=0.1,
left=0.10,
right=0.90,
hspace=0.0,
wspace=0.05)

fig.suptitle('Filter length statistics',y=0.98)


In [ ]:
vmin = np.min(dens_tot.value)
vmax = np.max(dens_tot.value)
norm = LogNorm(vmin=vmin,vmax=vmax)
extent = slicer.centered_extent.to('Mpc')

norm2 = Normalize(vmin=0.0,vmax=0.5)
cmap2 = plt.cm.gray
cmap2.set_over('white',alpha=0.0)

plt.figure(1)
plt.clf()

fig, axes = plt.subplots(1,3, figsize=(12,6),sharex=True, sharey=True)

im1 = axes[0].imshow(
    dens_tot.value, origin='lower', norm=norm,extent=extent.value)
im2 = axes[1].imshow(
    dens_bulk.value, origin='lower', norm=norm,extent=extent.value)
im3 = axes[2].imshow(
    dens_turb.value, origin='lower', norm=Normalize(vmin=-1.0*dens_turb.value.max(),vmax=1.0*dens_turb.value.max()),cmap="RdBu_r",extent=extent.value)

im4 = axes[0].imshow(
    slicer_iteration_converged, origin='lower', norm=norm2,extent=extent.value, cmap=cmap2)
im4 = axes[2].imshow(
    slicer_iteration_converged, origin='lower', norm=norm2,extent=extent.value, cmap=cmap2)

axes[0].set_title('Total')
axes[1].set_title('Bulk')
axes[2].set_title('Turbulent')

for i in range(3):
    axes[i].set_xlabel(extent.label('x'))

axes[0].set_ylabel(extent.label('y'))

# fig.subplots_adjust(bottom=0.2)
# cbar_ax = fig.add_axes([0.1, 0.15, 0.8, 0.05])

cbar = fig.colorbar(im1, orientation='horizontal')
cbar.set_label(dens_tot.label('\\rho'))

cbar = fig.colorbar(im2, orientation='horizontal')
cbar.set_label(dens_tot.label('\\rho'))

cbar = fig.colorbar(im3, orientation='horizontal')
cbar.set_label(dens_tot.label('\\rho'))
# cbar = plt.colorbar()
# fig.tight_layout(pad=1.00, h_pad=None, w_pad=None, rect=None)


fig.subplots_adjust(top=0.9,
bottom=0.1,
left=0.10,
right=0.90,
hspace=0.0,
wspace=0.05)

fig.suptitle('Where are the non-converged particles?\n'+'(black contours)',y=0.95)


In [ ]:
sf2 = tc.SmoothingFilter(snap, center, widths, npix=128, orientation=None, 
                        search_radius=5.*filter_length.value)

filt_density_non_iter = sf2.filter_variable('0_Density',filter_length, 
                                  weight='0_Volume', filter_type="gaussian", iterative=False)

snap['0_DensityTurbNonIter'] = snap['0_Density'].copy
snap['0_DensityTurbNonIter'][sf2.index] = snap['0_Density'][sf2.index] - filt_density_non_iter

dens_turb_non_iter = slicer.slice_variable('0_DensityTurbNonIter')
dens_turb_non_iter = dens_turb_non_iter.to_physical.cgs

In [ ]:
vmin = -0.3*dens_turb.value.max()
vmax = 0.3*dens_turb.value.max()
norm = Normalize(vmin=vmin,vmax=vmax)
cmap = cmr.fusion

plt.figure(1)
plt.clf()

fig, axes = plt.subplots(1,2, figsize=(10,6),sharex=True, sharey=True)

im1 = axes[0].imshow(
    dens_turb_non_iter.value, origin='lower', norm=norm,cmap=cmap,extent=extent.value)

im2 = axes[1].imshow(
    dens_turb.value, origin='lower', norm=norm,cmap=cmap,extent=extent.value)

axes[0].set_title('Non-iterative')
axes[1].set_title('Iterative')

for i in range(2):
    axes[i].set_xlabel(extent.label('x'))

axes[0].set_ylabel(extent.label('y'))

# fig.subplots_adjust(bottom=0.2)
# cbar_ax = fig.add_axes([0.1, 0.15, 0.8, 0.05])

cbar = fig.colorbar(im1, orientation='horizontal')
cbar.set_label(dens_tot.label('\\rho'))

cbar = fig.colorbar(im2, orientation='horizontal')
cbar.set_label(dens_tot.label('\\rho'))




fig.subplots_adjust(top=0.9,
bottom=0.1,
left=0.10,
right=0.90,
hspace=0.0,
wspace=0.05)

fig.suptitle('Turbulent density field (iterative vs non-iterative filter)',y=0.98)
